# CCOS — Creative Confidence Operating System
## Scoring Engine · Multi-Format Rubric Registry
### AVC Creative OS · Level 1 Prototype · v2

---

This notebook implements the **Creative Confidence Score (CCS)** using the Anthropic API.
The rubric registry now includes Social Static (original) and both YouTube video formats
(Long-Form and Shorts), grounded in platform algorithm research (Phase 3).

**What this notebook does:**
1. Accepts a creative asset (image or video frame) + optional caption copy
2. Routes it through the correct platform rubric via the registry
3. Scores it against 3 Synthetic Personas in parallel
4. Returns a weighted **Creative Confidence Score (0–100)**
5. Flags `consensus_warning` when persona verdict variance is high
6. Runs brand safety pre-scoring gate for video formats
7. Outputs a per-dimension heatmap and written critique

**Format scope (v2):**
- `social_static` — Meta Feed / Instagram Feed / LinkedIn Feed
- `youtube_longform` — YouTube long-form video + pre-roll/mid-roll ads
- `youtube_shorts` — YouTube Shorts (≤ 60s standard / up to 3 min)

**Model:** `claude-sonnet-4-20250514` (vision + text)  
**Thresholds:** ≥ 70 launch-ready · 55–69 needs revision · < 55 do-not-run

---

> **For Yena:** The rubric registry pattern (Cell 4) is the key architectural decision to review.
> New formats are added by writing a rubric definition only — no code changes required.
> The YouTube rubrics (v2) are grounded in Phase 3 Deep Research outputs — see the
> CCOS Platform Algorithm Research Brief in Notion for full sourcing.
> The `consensus_warning` (Cell 7) and brand safety pre-gate (new in Cell 4b) are
> the two non-negotiable guardrails before any asset reaches activation.


---
## Cell 1 — Install dependencies

In [ ]:
!pip install anthropic Pillow requests --quiet
print('Dependencies installed.')

---
## Cell 2 — Configuration

Set your Anthropic API key. In Colab, use **Secrets** (🔑 icon in left sidebar) and add `ANTHROPIC_API_KEY`.

In [ ]:
import os
import anthropic

# ── Colab Secrets (recommended) ──────────────────────────────────────────────
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    # ── Fallback: paste key directly (dev only, never commit) ────────────────
    api_key = os.environ.get('ANTHROPIC_API_KEY', 'YOUR_KEY_HERE')

client = anthropic.Anthropic(api_key=api_key)
MODEL  = 'claude-sonnet-4-20250514'

print(f'Client ready. Model: {MODEL}')

---
## Cell 3 — Load the creative asset

Choose one of three methods:
- **A** — Upload from local disk (Colab file picker)
- **B** — Load from a public URL
- **C** — Use the built-in demo asset (default, no upload needed)

In [ ]:
import base64
import requests
from PIL import Image
from io import BytesIO
import IPython.display as ipd

# ── CONFIG: set METHOD to 'A', 'B', or 'C' ───────────────────────────────────
METHOD = 'C'

# Optional: caption copy to score alongside the image
CAPTION_COPY = "Upgrade any room in an afternoon. Free delivery on orders over $75."

# ── Platform context ─────────────────────────────────────────────────────────
PLATFORM      = 'meta_feed'   # meta_feed | instagram_feed | linkedin_feed
INTENT_SIGNAL = 'warm'        # hot | warm | cold
BRAND_VOICE   = 'approachable and direct — outcome-led, avoids jargon'

# ─────────────────────────────────────────────────────────────────────────────

def encode_image(img_bytes: bytes, media_type: str = 'image/jpeg') -> dict:
    return {
        'type': 'image',
        'source': {
            'type': 'base64',
            'media_type': media_type,
            'data': base64.standard_b64encode(img_bytes).decode('utf-8')
        }
    }

image_block = None

if METHOD == 'A':
    from google.colab import files
    uploaded = files.upload()
    fname    = list(uploaded.keys())[0]
    mt       = 'image/png' if fname.endswith('.png') else 'image/jpeg'
    image_block = encode_image(uploaded[fname], mt)
    print(f'Loaded: {fname}')

elif METHOD == 'B':
    IMAGE_URL   = 'https://YOUR-IMAGE-URL-HERE.jpg'
    resp        = requests.get(IMAGE_URL, timeout=10)
    mt          = resp.headers.get('Content-Type', 'image/jpeg').split(';')[0]
    image_block = encode_image(resp.content, mt)
    print(f'Loaded from URL: {IMAGE_URL}')

else:  # METHOD == 'C' — demo asset
    # Generates a simple demo image so the notebook runs without any upload
    demo_img = Image.new('RGB', (1080, 1080), color='#F0EFE9')
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(demo_img)
    draw.rectangle([80, 80, 1000, 700], fill='#FFFFFF', outline='#D3D1C7', width=2)
    draw.rectangle([120, 120, 960, 660], fill='#E1F5EE')
    draw.text((540, 390), 'DEMO PRODUCT IMAGE', fill='#1D9E75', anchor='mm')
    draw.text((540, 780), CAPTION_COPY, fill='#1a1a18', anchor='mm')
    draw.rectangle([380, 860, 700, 920], fill='#1D9E75')
    draw.text((540, 890), 'Explore the range', fill='#FFFFFF', anchor='mm')
    buf = BytesIO()
    demo_img.save(buf, format='JPEG', quality=90)
    image_block = encode_image(buf.getvalue(), 'image/jpeg')
    ipd.display(demo_img.resize((400, 400)))
    print('Demo asset loaded. Replace with a real creative for calibrated scoring.')

print(f'Platform: {PLATFORM} | Intent: {INTENT_SIGNAL}')

---
## Cell 4 — The Rubric Registry

The CCOS uses a **config-driven rubric registry** — not hard-coded `if/else` format logic.
New formats are added by writing a rubric definition only. No code changes required.

**v2 additions:** `youtube_longform` and `youtube_shorts` rubrics, grounded in
Phase 3 Deep Research (May 2026). Both include a brand safety pre-scoring gate.

**Rubric registry:**
- `social_static` — original 4-dimension rubric
- `youtube_longform` — 5-dimension rubric + brand safety binary gate
- `youtube_shorts` — 5-dimension rubric + compliance binary gate

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional

@dataclass
class RubricDimension:
    name:          str
    weight:        float   # must sum to 1.0 across all dimensions
    description:   str
    scoring_guide: str     # injected into the evaluation prompt

@dataclass
class FormatRubric:
    format_id:         str
    format_name:       str
    modality:          str            # 'vision' | 'text' | 'multimodal'
    dimensions:        List[RubricDimension]
    platform_context:  str
    pre_scoring_gate:  Optional[str] = None  # binary check; None = no gate
    calibration_note:  str = ''


# ─────────────────────────────────────────────────────────────────────────────
# RUBRIC 1 — Social Static
# Original 4-dimension rubric. Meta Feed / Instagram Feed / LinkedIn Feed.
# ─────────────────────────────────────────────────────────────────────────────
SOCIAL_STATIC_RUBRIC = FormatRubric(
    format_id   = 'social_static',
    format_name = 'Social Static (Meta Feed / Instagram Feed / LinkedIn Feed)',
    modality    = 'vision',
    platform_context = (
        'The creative will appear in a social media feed. '
        'The viewer is scrolling at speed. '
        'The creative has approximately 1.5 seconds to earn attention before scrolling past.'
    ),
    calibration_note = 'MVP placeholder thresholds. Calibrate with real rejected assets in Alpha session.',
    dimensions = [
        RubricDimension(
            name          = 'stop_scroll_hierarchy',
            weight        = 0.35,
            description   = 'Stop-scroll visual hierarchy',
            scoring_guide = (
                'Score 0-100. Evaluate: Does the creative have a single dominant visual element '
                'that commands immediate attention within 1.5 seconds? Is there a clear visual '
                'hierarchy (hero > supporting > CTA)? Does the composition guide the eye '
                'intentionally? Does it differentiate from generic category creative? '
                '100 = instantly arresting, impossible to scroll past. '
                '50 = visible but forgettable. 0 = visually chaotic or invisible in-feed.'
            )
        ),
        RubricDimension(
            name          = 'text_image_ratio',
            weight        = 0.20,
            description   = 'Text-to-image ratio and legibility',
            scoring_guide = (
                'Score 0-100. Evaluate: Is the text-to-image ratio appropriate for the platform? '
                'Meta Feed penalises heavy text overlays. Is all on-image text legible at '
                'mobile screen size (min 14pt equivalent)? Does text placement respect safe zones? '
                '100 = perfect balance, all text crisp and placed. '
                '50 = readable but crowded or safe-zone borderline. '
                '0 = text-heavy, illegible at scale, or safe-zone violation.'
            )
        ),
        RubricDimension(
            name          = 'native_feel',
            weight        = 0.25,
            description   = 'Platform nativeness',
            scoring_guide = (
                'Score 0-100. Evaluate: Does the creative feel native to the platform it is '
                'being deployed on, or does it feel like a repurposed asset from another context? '
                'Does the aspect ratio, composition style, and tone match platform norms? '
                '100 = indistinguishable from organic content on that platform. '
                '50 = acceptable but clearly an ad. 0 = obviously imported from another format.'
            )
        ),
        RubricDimension(
            name          = 'caption_cta_alignment',
            weight        = 0.20,
            description   = 'Caption and CTA alignment to intent signal',
            scoring_guide = (
                'Score 0-100. Evaluate the caption copy and CTA (if visible). '
                'Is the CTA calibrated to the intent signal? '
                '(Hot = direct/transactional, Warm = inviting/low-pressure, Cold = curious/non-committal) '
                'Is the caption copy specific and outcome-led, or vague and aspirational? '
                '100 = precisely calibrated CTA, specific and compelling copy. '
                '50 = functional but generic. 0 = wrong intent register or no CTA.'
            )
        )
    ]
)


# ─────────────────────────────────────────────────────────────────────────────
# RUBRIC 2 — YouTube Long-Form Video
# 5-dimension rubric grounded in Phase 3 Deep Research (May 2026).
# Covers: long-form video + pre-roll / mid-roll ads.
# Pre-scoring gate: Brand Safety & Advertiser Compliance (binary).
# Key calibration benchmarks:
#   30-Second Hook Rate target : >= 70-80%
#   Relative watch time target : >= 85th cohort percentile
#   Ad view rate (skippable)   : >= 30%
#   Ad Quality Score target    : >= 7/10
#   Pattern interrupt cadence  : every 20-40 seconds
# ─────────────────────────────────────────────────────────────────────────────
YOUTUBE_LONGFORM_RUBRIC = FormatRubric(
    format_id   = 'youtube_longform',
    format_name = 'YouTube Long-Form Video (+ pre-roll / mid-roll ads)',
    modality    = 'vision',
    pre_scoring_gate = (
        'BRAND SAFETY PRE-SCORING GATE (binary — must pass before dimension scoring). '
        'Score 0 and return for revision if any of the following are present: '
        '(1) Content that would be flagged by YouTube borderline classifier '
        '(sensationalism, misinformation, polarising claims without authoritative sourcing). '
        '(2) Muted or silent audio in an in-stream ad placement. '
        '(3) Metadata that does not match the spoken content of the video '
        '(tags or description containing terms not spoken in the video). '
        '(4) For ads: landing page that does not match the creative promise or fails Core Web Vitals. '
        'If all checks pass, proceed to dimension scoring.'
    ),
    platform_context = (
        'YouTube long-form video operates across three distinct discovery surfaces, each with a '
        'different retrieval architecture and intent model: '
        'Search (Warm-Hot intent: cosine similarity between query embedding and document embedding). '
        'Browse/Homepage (Cold-Warm intent: sequential transformer with Long-Term Embedding prefix token). '
        'Suggested/Up Next (Hot intent: hybrid co-visitation graph + Trans-Topics supervised learning). '
        'The algorithm evaluates implicit feedback (watch time, completion) with far higher trust than '
        'explicit feedback (likes, comments, shares). '
        'Relative watch time at the 85th percentile of the length cohort triggers expanded distribution. '
        'Below cohort mean triggers suppression.'
    ),
    calibration_note = (
        'Benchmarks grounded in Phase 3 Deep Research (May 2026). '
        '30-Second Hook Rate >= 70-80%. Relative watch time >= 85th cohort percentile. '
        'Ad skippable view rate >= 30%. Ad Quality Score >= 7/10. '
        'Pattern interrupts every 20-40 seconds. Metadata-to-transcript consistency 100%.'
    ),
    dimensions = [
        RubricDimension(
            name          = 'thumbnail_strength',
            weight        = 0.15,
            description   = 'Thumbnail stop-scroll strength (pre-click hook)',
            scoring_guide = (
                'Score 0-100. Evaluate the thumbnail as a pre-click hook. '
                'The algorithm uses Model-Based Debiasing: CTR is measured as a z-score within '
                'the videos length cohort — must reach 85th percentile for expanded distribution. '
                'CRITICAL: The Clickbait Video Detector (CVD) cross-references CTR against '
                'Average View Duration. A thumbnail that drives high CTR but does not convert '
                'to watch time is penalised more severely than a weak thumbnail. '
                'Score high if: specific deliverable promise made, no misleading overlays or '
                'false urgency, visual hierarchy is clean and platform-appropriate. '
                'Score low if: thumbnail overpromises relative to the video content, uses '
                'extreme facial expressions or clickbait patterns, or makes a promise the '
                'first 30 seconds cannot deliver. '
                '100 = cohort-competitive, specific promise, zero CVD-detectable patterns. '
                '50 = adequate but generic. 0 = CVD-detectable clickbait.'
            )
        ),
        RubricDimension(
            name          = 'opening_hook_retention',
            weight        = 0.20,
            description   = 'Opening hook retention (0-5s / first 30s)',
            scoring_guide = (
                'Score 0-100. Target: 30-Second Hook Rate >= 70-80%. '
                'Evaluate: Does the opening prevent early drop-off in the seed cohort? '
                'Does it deliver on the thumbnail promise within the first 5 seconds? '
                'Does it speak at least one primary keyword within the first 30 seconds '
                '(for speech-to-text semantic indexing on Search and Suggested surfaces)? '
                'Are all animated logos, branding cards, spinning intros, and generic '
                'welcome sequences absent? '
                'Suppression triggers: overlong intros, delayed payoff (thumbnail promise '
                'not previewed in first 5s), robotic/synthetic voice delivery. '
                '100 = cold open, immediate promise delivery, primary keyword spoken, '
                'predicted Hook Rate >= 70%. '
                '50 = acceptable intro with minor delay. '
                '0 = animated logo, generic intro, or flat synthetic narration.'
            )
        ),
        RubricDimension(
            name          = 'watch_time_architecture',
            weight        = 0.25,
            description   = 'Watch time architecture (pacing + narrative structure)',
            scoring_guide = (
                'Score 0-100. Target: relative watch time >= 85th percentile of length cohort. '
                'The algorithm uses MBD (Model-Based Debiasing) to calibrate watch time as a '
                'z-score within the videos duration cohort. Implicit watch signals are trusted '
                'far more than explicit engagement (likes, comments). '
                'Evaluate based on retention graph archetype likelihood: '
                'Flatline (stable retention) = expanded distribution. '
                'Straight Decline (sharp early drop) = accelerated suppression. '
                'Spikes (rewatch moments) = boosts Suggested score. '
                'Dips (fast-forward zones) = demotes specific timestamps. '
                'Pacing benchmark: pattern interrupt (zoom, b-roll, text overlay, audio '
                'transition) every 20-40 seconds maintains watch time above cohort mean. '
                '100 = zero dead space, pattern interrupts present, Flatline/Spike archetype '
                'likely, predicted cohort percentile >= 85th. '
                '50 = some padding, minor dip zones, 50-70th percentile likely. '
                '0 = Straight Decline archetype likely, suppression triggered.'
            )
        ),
        RubricDimension(
            name          = 'contextual_surface_alignment',
            weight        = 0.25,
            description   = 'Contextual surface alignment (Search / Browse / Suggested)',
            scoring_guide = (
                'Score 0-100. Evaluate semantic fit with the intended discovery surface. '
                'Misalignment causes elimination before ranking — the creative is never '
                'scored on engagement signals at all. '
                'Search surface (Warm-Hot): Does the videos metadata and spoken content '
                'semantically map via cosine similarity to the target search queries? '
                'Browse surface (Cold-Warm): Is the topic and tone consistent with the '
                'target audiences Long-Term Embedding (demonstrated long-term watch history)? '
                'Suggested surface (Hot): Does the topic connect via Trans-Topics transition '
                'probability to adjacent content in the category? '
                'LTE anchoring: Creative contradicting the users long-term embedding is ranked '
                'down even if it matches immediate session context. '
                'Also evaluate: Is the content free of borderline signals that would restrict '
                'it to Search and Subscriptions only (excluding Browse and Suggested)? '
                '100 = strong semantic fit across all relevant surfaces, no borderline risk. '
                '50 = adequate fit for one surface, weak on others. '
                '0 = borderline-flagged content or complete surface misalignment.'
            )
        ),
        RubricDimension(
            name          = 'intent_calibrated_cta',
            weight        = 0.15,
            description   = 'Intent-calibrated CTA and messaging',
            scoring_guide = (
                'Score 0-100. Evaluate CTA and messaging calibration to the intent surface. '
                'Hot / Suggested: CTA must support session continuation (watch next, subscribe) '
                'not session exit (purchase, sign-up). Algorithm optimises for session depth. '
                'Warm / Search: Creative must immediately confirm it answers the search query '
                'within first 5 seconds. High bounce rate demotes future search ranking for query. '
                'Cold / Browse: Low-friction engagement CTA only. Conversion asks have no '
                'established relationship context. LTE alignment more important than CTA strength. '
                'For ads: skippable view rate target >= 30% (5-second skip gate). '
                'Ad Quality Score is the divisor in CPC: Actual CPC = (Ad Rank below / Quality Score) + $0.01. '
                'Higher Quality Score directly reduces CPV. Muted audio = confirmed demotion. '
                '100 = precisely calibrated to surface and intent, strong view rate prediction. '
                '50 = adequate but generic CTA for context. '
                '0 = wrong intent register (e.g. hard conversion ask in Cold Browse).'
            )
        )
    ]
)


# ─────────────────────────────────────────────────────────────────────────────
# RUBRIC 3 — YouTube Shorts
# 5-dimension rubric grounded in Phase 3 Deep Research (May 2026).
# Covers: short-form vertical video (<= 60s standard / up to 3 min).
# Pre-scoring gate: Platform Compliance (binary — silent suppression triggers).
# Key calibration benchmarks:
#   VVSA (Viewed vs Swiped Away) target : >= 70%
#   Completion rate target              : >= 80%
#   Optimal video length                : 15-35 seconds
#   Replay rate target                  : > 15%
#   Compliance                          : binary — any watermark or
#                                         Content ID violation = suppression
# ─────────────────────────────────────────────────────────────────────────────
YOUTUBE_SHORTS_RUBRIC = FormatRubric(
    format_id   = 'youtube_shorts',
    format_name = 'YouTube Shorts (vertical video <= 60s / up to 3 min)',
    modality    = 'vision',
    pre_scoring_gate = (
        'PLATFORM COMPLIANCE PRE-SCORING GATE (binary — silent suppression triggers). '
        'Score 0 and return for revision if any of the following are detected: '
        '(1) External platform watermark visible in any frame '
        '(TikTok, Instagram Reels logos — detected via frame scanning). '
        '(2) Reused or duplicate content (audio-visual fingerprint match). '
        '(3) Metadata repetition/spam (identical titles or hashtag sets across multiple Shorts). '
        '(4) For Shorts 1-3 minutes: any Content ID claim present (results in global block). '
        '(5) Aspect ratio not 9:16 or resolution below 1080x1920. '
        'Note: Suppression is SILENT (70-90% impression drop, no notification). '
        'If all checks pass, proceed to dimension scoring.'
    ),
    platform_context = (
        'YouTube Shorts is a categorically different algorithmic system from long-form YouTube. '
        'It uses an Explore/Exploit pipeline: a seed cohort tests the Short, and if behavioral '
        'metrics exceed thresholds the system scales to wider demographic rings. '
        'Primary metric: relative completion rate (%) — NOT absolute watch time. '
        'Hook window: 1-2 seconds (vs 3-5 seconds for long-form). '
        'Intent model: passive inference from last 5-10 swipe interactions — not explicit. '
        'Loop/replay compounds the retention score: each replay = positive engagement signal. '
        'Content lifespan: 28-30 day freshness decay cliff. '
        'Suppression is silent: 70-90% impression drop with no notification.'
    ),
    calibration_note = (
        'Benchmarks grounded in Phase 3 Deep Research (May 2026). '
        'VVSA >= 70% (top-tier), 60-70% (average), 50-60% (substandard), <50% (suppressed). '
        'Completion rate target >= 80%. Optimal length 15-35 seconds. '
        'Replay rate target > 15%. Shorts Ranking Velocity proportional to '
        'Completion Rate x Video Length.'
    ),
    dimensions = [
        RubricDimension(
            name          = 'hook_velocity',
            weight        = 0.30,
            description   = 'Hook velocity (first 1-2 seconds / VVSA gate)',
            scoring_guide = (
                'Score 0-100. Target VVSA (Viewed vs Swiped Away) >= 70%. '
                'There is no thumbnail in Shorts. The first frame IS the hook. '
                'VVSA is the definitive first-stage ranking signal. Below 50% = distribution halted. '
                'Suppression triggers (cause up to 70% immediate swipe-away): '
                'verbal intros (Hey guys, welcome back...), static branding logos, '
                'any visual or auditory dead space in first 2 seconds. '
                'Score high if: first frame leads with high visual movement or mid-action start, '
                'dynamic text overlay establishing a curiosity gap, no intro of any kind. '
                'VVSA tiers: >=70% = top-tier (100 pts), 60-70% = average (70 pts), '
                '50-60% = substandard (50 pts), <50% = suppression triggered (0 pts). '
                '100 = immediate pattern interrupt, no dead space, predicted VVSA >= 70%. '
                '0 = verbal intro or static logo in first 2 seconds.'
            )
        ),
        RubricDimension(
            name          = 'completion_rate_architecture',
            weight        = 0.25,
            description   = 'Completion rate architecture (pacing + length + loop engineering)',
            scoring_guide = (
                'Score 0-100. Target: >= 80% completion rate. '
                'Relative completion rate (%) is the primary ranking signal for the Exploit pipeline. '
                'Confirmed formula: Shorts Ranking Velocity proportional to Completion Rate x Video Length. '
                'Optimal length: 15-35 seconds (balances completion potential with narrative depth). '
                'A 45-second Short at 70% completion outranks a 15-second Short at 40% completion. '
                'Loop engineering: seamless loop (final 2s connects to first 2s visually, verbally, '
                'or tonally) multiplies retention score. Each replay = positive engagement signal. '
                'Target replay rate > 15%. '
                '100 = length 15-35s, zero dead space, loop engineered, predicted completion >= 80%. '
                '50 = good pacing, minor padding, adequate length. '
                '0 = slow pacing, excessive length without retention engineering, abrupt ending.'
            )
        ),
        RubricDimension(
            name          = 'platform_nativeness_compliance',
            weight        = 0.20,
            description   = 'Platform nativeness and compliance',
            scoring_guide = (
                'Score 0-100. Evaluate native format adherence and multimodal classification fit. '
                'The Shorts multimodal parser scans visual frames, audio waveforms, and transcripts '
                'before any human interactions are registered. Content that looks repurposed from '
                'another format is misclassified or penalised. '
                'Three nativeness layers: '
                '(1) Format nativeness: vertical, fast-paced, no external watermarks, native length. '
                '(2) Sequential context: topically relevant to a viewer mid-session on similar content. '
                '(3) Loop architecture: final frame connects to first frame for seamless replay. '
                'Also evaluate: does the content feel natively produced for Shorts, or does it '
                'look like repurposed long-form, static, or TikTok content? '
                '100 = fully native format, zero repurposed signals, loop-engineered. '
                '50 = acceptable but identifiable as repurposed or non-native. '
                '0 = obviously repurposed from another platform or format.'
            )
        ),
        RubricDimension(
            name          = 'contextual_sequential_alignment',
            weight        = 0.15,
            description   = 'Contextual and sequential alignment',
            scoring_guide = (
                'Score 0-100. Evaluate topical fit with the viewers passive session context. '
                'The Shorts feed uses sequential modeling based on the last 5-10 rapid interactions. '
                'Unlike long-form YouTube, there is no explicit search query or thumbnail click. '
                'Intent is inferred passively from session behaviour. '
                'A Short that is contextually misaligned with the viewers recent swipe history '
                'causes negative VVSA regardless of its intrinsic quality. '
                'Evaluate: does the topic, tone, and format fit cleanly within a session stream '
                'of similar content? Does it avoid jarring tonal or topical discontinuity? '
                '100 = clearly fits a content category stream, high sequential alignment. '
                '50 = broad enough to fit multiple contexts but not specifically native to any. '
                '0 = tonally or topically jarring relative to its content category.'
            )
        ),
        RubricDimension(
            name          = 'intent_calibrated_cta_shorts',
            weight        = 0.10,
            description   = 'Intent-calibrated CTA (passive intent model)',
            scoring_guide = (
                'Score 0-100. Evaluate CTA appropriateness for the passive intent model. '
                'Shorts uses passive intent inference from session behaviour — NOT explicit signals. '
                'The only native conversion mechanism is the related link editor (paid: CTA button). '
                'Verbal CTAs to external URLs are not clickable in the Shorts feed. '
                'Cold (Attract): CTA should drive completion and subscription, not conversion. '
                'Hard conversion asks have no contextual grounding and cause negative VVSA. '
                'Warm (Nurture): CTAs inviting comment engagement drive the 3-hour engagement '
                'velocity signal (Which would you choose? Tell me in the comments). '
                'Hot (Convert): Related link editor click-through is the only supported mechanism. '
                'Verbal CTAs to external URLs are not clickable in the Shorts feed. '
                '100 = CTA is native to the intent state, uses supported mechanisms. '
                '50 = adequate but generic CTA. '
                '0 = hard conversion ask in Cold context, or verbal URL CTA (not clickable).'
            )
        )
    ]
)


# ── Rubric Registry ───────────────────────────────────────────────────────────
# Add new formats here — no other code changes required.
# Research status: Social Static (original) · YouTube Long-Form (v2) · YouTube Shorts (v1)
# Pending Phase 3 research: Meta · Instagram · LinkedIn · TikTok · Display Banner · Long-form
RUBRIC_REGISTRY: Dict[str, FormatRubric] = {
    'social_static':    SOCIAL_STATIC_RUBRIC,
    'youtube_longform': YOUTUBE_LONGFORM_RUBRIC,
    'youtube_shorts':   YOUTUBE_SHORTS_RUBRIC,
    # 'meta_feed':        META_FEED_RUBRIC,        # Phase 3 research pending
    # 'instagram_reels':  INSTAGRAM_REELS_RUBRIC,  # Phase 3 research pending
    # 'linkedin_feed':    LINKEDIN_FEED_RUBRIC,     # Phase 3 research pending
    # 'tiktok':           TIKTOK_RUBRIC,            # Phase 3 reference
    # 'display_banner':   BANNER_RUBRIC,            # Phase 2 build
    # 'long_form_text':   LONG_FORM_RUBRIC,         # Phase 2 build
}

def route_to_rubric(format_type: str) -> FormatRubric:
    """Format Router — reads format_type tag, returns correct rubric from registry."""
    if format_type not in RUBRIC_REGISTRY:
        raise ValueError(
            f"Format '{format_type}' not in rubric registry. "
            f"Available: {list(RUBRIC_REGISTRY.keys())}"
        )
    return RUBRIC_REGISTRY[format_type]

def check_pre_scoring_gate(rubric: FormatRubric) -> None:
    """Print the pre-scoring gate check if one is defined for this rubric."""
    if rubric.pre_scoring_gate:
        print(f'PRE-SCORING GATE ACTIVE for {rubric.format_name}')
        print('-' * 60)
        print(rubric.pre_scoring_gate)
        print('-' * 60)
        print('Confirm all checks pass before proceeding to dimension scoring.')
    else:
        print(f'No pre-scoring gate for {rubric.format_name}. Proceed to dimensions.')

# ── Demo: load a rubric and inspect it ───────────────────────────────────────
# Change FORMAT_TYPE to score a different format.
# Options: 'social_static' | 'youtube_longform' | 'youtube_shorts'
FORMAT_TYPE = 'social_static'

rubric = route_to_rubric(FORMAT_TYPE)
print(f'Rubric loaded : {rubric.format_name}')
print(f'Dimensions    : {[d.name for d in rubric.dimensions]}')
print(f'Weights sum   : {sum(d.weight for d in rubric.dimensions):.2f}  (must be 1.0)')
print(f'Modality      : {rubric.modality}')
print(f'Pre-gate      : {"Yes" if rubric.pre_scoring_gate else "No"}')
if rubric.calibration_note:
    print(f'Calibration   : {rubric.calibration_note}')
print()
check_pre_scoring_gate(rubric)


---
## Cell 5 — Synthetic Persona Generator

Three personas per evaluation — one per intent signal (Hot / Warm / Cold). In production these are built from client first-party data in the Clean Room. Here they are constructed from the brief context as a Bridge evaluation.

In [ ]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class SyntheticPersona:
    intent_signal:  str
    decision_stage: str
    scroll_speed:   str
    pause_triggers: str
    friction_points: str
    cta_sensitivity: str
    vocabulary_note: str

PERSONAS = [
    SyntheticPersona(
        intent_signal   = 'hot',
        decision_stage  = 'In-market, active purchase intent. Has researched the category. Ready to buy if the right offer appears.',
        scroll_speed    = 'Fast — scanning for a reason to stop.',
        pause_triggers  = 'Specific product visible, price or offer clarity, proof element (reviews, ratings, delivery promise).',
        friction_points = 'Vague claims, aspirational lifestyle without product, no clear next step.',
        cta_sensitivity = 'Responds to direct transactional CTAs. Rejects pressure tactics.',
        vocabulary_note = 'Responds to: specific, proven, guaranteed, ships today. Alienated by: journey, curated, elevate.'
    ),
    SyntheticPersona(
        intent_signal   = 'warm',
        decision_stage  = 'Aware of the category, actively considering. Has not committed to a brand.',
        scroll_speed    = 'Medium — will pause for something visually interesting or relevant.',
        pause_triggers  = 'Clear outcome visible, relatable scenario, brand differentiation, specific benefit.',
        friction_points = 'Hard sell, urgency/scarcity language, overly polished lifestyle imagery that feels unattainable.',
        cta_sensitivity = 'Responds to inviting low-pressure CTAs. Rejects urgency. Needs a reason before committing.',
        vocabulary_note = 'Responds to: explore, discover, see how, learn more. Alienated by: limited time, act now, last chance.'
    ),
    SyntheticPersona(
        intent_signal   = 'cold',
        decision_stage  = 'Unaware or passively adjacent to the category. Not actively looking.',
        scroll_speed    = 'Fast — low tolerance for anything that feels like an ad.',
        pause_triggers  = 'Visually distinctive, cultural relevance, genuine curiosity trigger, feels organic not advertised.',
        friction_points = 'Anything that feels like an ad. Product-first composition. Transactional language.',
        cta_sensitivity = 'Low. Rejects all transactional CTAs. Needs pure curiosity or cultural hook first.',
        vocabulary_note = 'Responds to: stories, ideas, perspectives. Alienated by: any purchase language, superlatives.'
    )
]

def build_persona_context(persona: SyntheticPersona) -> str:
    return f"""You are evaluating this creative as a Synthetic Persona with the following profile:

Intent signal: {persona.intent_signal.upper()}
Decision stage: {persona.decision_stage}
Scroll speed: {persona.scroll_speed}
What makes you pause: {persona.pause_triggers}
What creates friction: {persona.friction_points}
CTA sensitivity: {persona.cta_sensitivity}
Vocabulary note: {persona.vocabulary_note}

You are not a marketing expert. You are this specific person encountering this creative in their feed.
Evaluate from their perspective, not from a brand strategy perspective."""

print(f'3 Synthetic Personas loaded: {[p.intent_signal for p in PERSONAS]}')

---
## Cell 6 — The Evaluation Engine

Calls the Anthropic API for each persona × each rubric dimension. Returns structured JSON scores.
Three personas evaluated in sequence (parallel execution via `asyncio` in production).

In [ ]:
import json
import re
import time

def build_evaluation_prompt(
    rubric:    FormatRubric,
    persona:   SyntheticPersona,
    platform:  str,
    intent:    str,
    brand_voice: str,
    caption:   Optional[str] = None
) -> str:

    dims_block = '\n'.join([
        f'DIMENSION {i+1}: {d.name}\n{d.scoring_guide}'
        for i, d in enumerate(rubric.dimensions)
    ])

    caption_block = f'\nCaption copy to evaluate alongside the image:\n"{caption}"\n' if caption else ''

    return f"""You are evaluating a social media creative asset for the CCOS (Creative Confidence Operating System).

{build_persona_context(persona)}

EVALUATION CONTEXT
Platform: {platform}
Campaign intent signal: {intent.upper()}
Brand voice: {brand_voice}
{rubric.platform_context}
{caption_block}
SCORING RUBRIC — score each dimension 0–100:
{dims_block}

INSTRUCTIONS
1. Examine the image carefully from the perspective of this specific persona.
2. Score each dimension independently. Do not anchor scores to each other.
3. For each dimension, write a 1–2 sentence critique explaining the score.
4. Write an overall assessment (2–3 sentences) from this persona's perspective.
5. State your verdict: launch_ready (≥70 weighted) | needs_revision (55–69) | do_not_run (<55)

RESPONSE FORMAT — return ONLY valid JSON, no preamble, no markdown fences:
{{
  "persona_intent": "{persona.intent_signal}",
  "dimension_scores": {{
    {', '.join([f'"{d.name}": {{"score": <int>, "critique": "<string>"}}' for d in rubric.dimensions])}
  }},
  "overall_assessment": "<string>",
  "verdict": "launch_ready | needs_revision | do_not_run"
}}"""


def score_for_persona(
    persona:     SyntheticPersona,
    image_block: dict,
    rubric:      FormatRubric,
    platform:    str,
    intent:      str,
    brand_voice: str,
    caption:     Optional[str] = None
) -> dict:
    prompt = build_evaluation_prompt(rubric, persona, platform, intent, brand_voice, caption)

    response = client.messages.create(
        model      = MODEL,
        max_tokens = 1000,
        messages   = [{
            'role': 'user',
            'content': [
                image_block,
                {'type': 'text', 'text': prompt}
            ]
        }]
    )

    raw = response.content[0].text.strip()
    # Strip markdown fences if present
    raw = re.sub(r'^```json\s*|^```\s*|```$', '', raw, flags=re.MULTILINE).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        print(f'  Warning: JSON parse failed for {persona.intent_signal} persona. Raw response:\n{raw[:300]}')
        return {'persona_intent': persona.intent_signal, 'error': 'parse_failed', 'raw': raw}


print('Evaluation engine ready.')
print(f'Scoring {len(PERSONAS)} personas × {len(rubric.dimensions)} dimensions = {len(PERSONAS) * len(rubric.dimensions)} API evaluations')

---
## Cell 7 — Run the evaluation + compute CCS

Scores all three personas, computes the weighted Creative Confidence Score, and fires the `consensus_warning` if persona verdicts diverge.

In [ ]:
VERDICT_MAP = {'launch_ready': 2, 'needs_revision': 1, 'do_not_run': 0}

def compute_weighted_score(persona_result: dict, rubric: FormatRubric) -> float:
    """Compute weighted CCS for a single persona result."""
    total = 0.0
    dim_scores = persona_result.get('dimension_scores', {})
    for dim in rubric.dimensions:
        entry = dim_scores.get(dim.name, {})
        score = entry.get('score', 50) if isinstance(entry, dict) else 50
        total += score * dim.weight
    return round(total, 1)


def check_consensus_warning(persona_results: list) -> dict:
    """
    Fires consensus_warning when persona verdicts diverge significantly.
    Critical guardrail: the smoothed score must not obscure high-variance persona disagreement.
    """
    verdicts = [VERDICT_MAP.get(r.get('verdict', 'needs_revision'), 1) for r in persona_results]
    verdict_range = max(verdicts) - min(verdicts)
    scores  = [r.get('_weighted_score', 60) for r in persona_results]
    score_std = (sum((s - sum(scores)/len(scores))**2 for s in scores) / len(scores)) ** 0.5

    # Warning fires when: verdicts span all three levels, OR score std dev > 12
    warning = verdict_range >= 2 or score_std > 12

    return {
        'consensus_warning': warning,
        'verdict_range':     verdict_range,
        'score_std_dev':     round(score_std, 1),
        'verdicts':          [r.get('verdict') for r in persona_results],
        'warning_reason':    (
            'Personas produced significantly divergent verdicts. '
            'Review per-persona critiques before activation decision.'
            if warning else None
        )
    }


# ── Run evaluation ────────────────────────────────────────────────────────────
print('Running CCOS evaluation...\n')
persona_results = []

for i, persona in enumerate(PERSONAS):
    print(f'  Scoring {persona.intent_signal.upper()} persona ({i+1}/3)...')
    result = score_for_persona(
        persona     = persona,
        image_block = image_block,
        rubric      = rubric,
        platform    = PLATFORM,
        intent      = INTENT_SIGNAL,
        brand_voice = BRAND_VOICE,
        caption     = CAPTION_COPY
    )
    result['_weighted_score'] = compute_weighted_score(result, rubric)
    persona_results.append(result)
    time.sleep(0.5)  # brief pause between calls

# ── Persona Consensus Factor ──────────────────────────────────────────────────
# CCS = average of per-persona weighted scores, modified by consensus factor.
# In production: each persona carries a client-calibrated multiplier.
# Here: equal weighting (1/3 each) as placeholder.
persona_scores = [r['_weighted_score'] for r in persona_results]
raw_ccs        = round(sum(persona_scores) / len(persona_scores), 1)

# ── Consensus Warning ─────────────────────────────────────────────────────────
consensus = check_consensus_warning(persona_results)

# ── Threshold classification ──────────────────────────────────────────────────
if raw_ccs >= 70:
    threshold_label = 'LAUNCH-READY'
    threshold_action = 'Proceed to activation'
elif raw_ccs >= 55:
    threshold_label = 'NEEDS REVISION'
    threshold_action = 'Return with critique'
else:
    threshold_label = 'DO NOT RUN'
    threshold_action = 'Rebuild'

print(f'\nEvaluation complete.')
print(f'Per-persona scores: {dict(zip([p.intent_signal for p in PERSONAS], persona_scores))}')
print(f'CCS: {raw_ccs}  |  {threshold_label}')
if consensus['consensus_warning']:
    print(f'⚠  CONSENSUS WARNING ACTIVE — {consensus["warning_reason"]}')

---
## Cell 8 — Render the full CCOS output report

In [ ]:
from IPython.display import display, HTML

def render_score_bar(score: int, width: int = 40) -> str:
    filled = int(score / 100 * width)
    bar    = '█' * filled + '░' * (width - filled)
    return f'[{bar}] {score}'

def render_ccs_report(
    rubric:          FormatRubric,
    persona_results: list,
    ccs:             float,
    threshold_label: str,
    threshold_action: str,
    consensus:       dict
):
    verdict_emoji = {'launch_ready': '✅', 'needs_revision': '⚠️', 'do_not_run': '🚫'}
    threshold_color = {'LAUNCH-READY': '#1D9E75', 'NEEDS REVISION': '#BA7517', 'DO NOT RUN': '#A32D2D'}

    html = f"""
    <style>
      .ccos-report {{ font-family: -apple-system, sans-serif; max-width: 860px; color: #1a1a18; }}
      .ccos-header {{ background: #f7f6f3; border: 1px solid #D3D1C7; border-radius: 10px;
                      padding: 20px 24px; margin-bottom: 20px; }}
      .ccos-score  {{ font-size: 52px; font-weight: 700;
                      color: {threshold_color[threshold_label]}; line-height: 1; }}
      .ccs-label   {{ font-size: 13px; font-weight: 600; letter-spacing: .5px;
                      color: {threshold_color[threshold_label]}; margin-top: 4px; }}
      .ccs-action  {{ font-size: 13px; color: #6b6a64; margin-top: 2px; }}
      .warn-banner {{ background: #FAEEDA; border: 1px solid #EF9F27; border-radius: 8px;
                      padding: 12px 16px; margin-bottom: 20px; font-size: 13px; color: #633806; }}
      .persona-card {{ background: #ffffff; border: 1px solid #D3D1C7; border-radius: 10px;
                       padding: 16px 20px; margin-bottom: 12px; }}
      .persona-title {{ font-size: 13px; font-weight: 600; text-transform: uppercase;
                         letter-spacing: .5px; color: #6b6a64; margin-bottom: 12px; }}
      .dim-row {{ display: flex; align-items: center; gap: 12px;
                   margin-bottom: 6px; font-size: 13px; }}
      .dim-name {{ width: 200px; color: #6b6a64; flex-shrink: 0; }}
      .dim-bar  {{ flex: 1; height: 6px; background: #f0efe9; border-radius: 3px; overflow: hidden; }}
      .dim-fill {{ height: 100%; border-radius: 3px; background: #1D9E75; }}
      .dim-score {{ width: 30px; text-align: right; font-weight: 500; }}
      .dim-critique {{ font-size: 12px; color: #9b9a94; margin-left: 212px;
                        margin-bottom: 10px; line-height: 1.4; font-style: italic; }}
      .assessment {{ background: #f7f6f3; border-radius: 6px; padding: 10px 14px;
                      font-size: 13px; color: #444441; line-height: 1.55;
                      margin-top: 10px; border-left: 3px solid #1D9E75; }}
      .persona-score {{ font-size: 22px; font-weight: 700; float: right;
                         color: {threshold_color[threshold_label]}; }}
      table {{ width: 100%; border-collapse: collapse; margin-top: 20px; font-size: 13px; }}
      th {{ background: #f0efe9; padding: 8px 12px; text-align: left;
             font-weight: 600; border-bottom: 1px solid #D3D1C7; }}
      td {{ padding: 8px 12px; border-bottom: 1px solid #f0efe9; }}
    </style>

    <div class="ccos-report">
      <div class="ccos-header">
        <div style="display:flex; justify-content:space-between; align-items:flex-start;">
          <div>
            <div style="font-size:11px; font-weight:600; letter-spacing:.8px;
                        color:#9b9a94; text-transform:uppercase; margin-bottom:6px;">
              CCOS · Creative Confidence Score · {rubric.format_name}
            </div>
            <div class="ccos-score">{ccs}</div>
            <div class="ccs-label">{threshold_label}</div>
            <div class="ccs-action">{threshold_action}</div>
          </div>
          <div style="text-align:right; font-size:12px; color:#9b9a94;">
            Platform: {PLATFORM.replace('_',' ')}<br>
            Intent: {INTENT_SIGNAL.upper()}<br>
            Personas: {len(persona_results)}<br>
            Gate version: bridge_v1
          </div>
        </div>
      </div>
    """

    if consensus['consensus_warning']:
        html += f"""
      <div class="warn-banner">
        ⚠ <strong>Consensus Warning</strong> — {consensus['warning_reason']}<br>
        <span style="font-size:12px;">Verdicts: {' · '.join(consensus['verdicts'])} &nbsp;|&nbsp;
        Score std dev: {consensus['score_std_dev']} &nbsp;|&nbsp;
        Verdict range: {consensus['verdict_range']}</span>
      </div>"""

    for result in persona_results:
        p_intent = result.get('persona_intent', '?').upper()
        p_score  = result.get('_weighted_score', 0)
        p_verdict = result.get('verdict', 'needs_revision')
        p_emoji  = verdict_emoji.get(p_verdict, '')
        dim_scores = result.get('dimension_scores', {})

        html += f"""
      <div class="persona-card">
        <div class="persona-title">
          {p_emoji} {p_intent} Persona &nbsp;&nbsp;
          <span style="font-weight:400; font-size:12px; color:#9b9a94;">{p_verdict.replace('_',' ')}</span>
          <span class="persona-score">{p_score}</span>
        </div>"""

        for dim in rubric.dimensions:
            entry    = dim_scores.get(dim.name, {})
            score    = entry.get('score', 0) if isinstance(entry, dict) else 0
            critique = entry.get('critique', '') if isinstance(entry, dict) else ''
            html += f"""
        <div class="dim-row">
          <span class="dim-name">{dim.description}</span>
          <div class="dim-bar"><div class="dim-fill" style="width:{score}%"></div></div>
          <span class="dim-score">{score}</span>
        </div>
        <div class="dim-critique">{critique}</div>"""

        assessment = result.get('overall_assessment', '')
        html += f"""
        <div class="assessment">{assessment}</div>
      </div>"""

    # Summary heatmap table
    html += '<table><tr><th>Dimension</th><th>Weight</th>'
    for p in persona_results:
        html += f'<th>{p.get("persona_intent","").upper()} score</th>'
    html += '<th>Avg</th></tr>'

    for dim in rubric.dimensions:
        scores_for_dim = []
        html += f'<tr><td>{dim.description}</td><td style="color:#9b9a94">{int(dim.weight*100)}%</td>'
        for r in persona_results:
            entry = r.get('dimension_scores', {}).get(dim.name, {})
            s = entry.get('score', 0) if isinstance(entry, dict) else 0
            scores_for_dim.append(s)
            color = '#1D9E75' if s >= 70 else '#BA7517' if s >= 55 else '#A32D2D'
            html += f'<td style="font-weight:500; color:{color}">{s}</td>'
        avg = round(sum(scores_for_dim) / len(scores_for_dim))
        html += f'<td style="font-weight:600">{avg}</td></tr>'

    html += f'<tr style="background:#f7f6f3; font-weight:600;"><td>CCS (weighted)</td><td></td>'
    for p in persona_results:
        html += f'<td>{p.get("_weighted_score",0)}</td>'
    html += f'<td style="color:{threshold_color[threshold_label]}; font-size:16px;">{ccs}</td></tr>'
    html += '</table></div>'

    display(HTML(html))


render_ccs_report(
    rubric           = rubric,
    persona_results  = persona_results,
    ccs              = raw_ccs,
    threshold_label  = threshold_label,
    threshold_action = threshold_action,
    consensus        = consensus
)

---
## Cell 9 — Export raw results as JSON

In [ ]:
import json
from datetime import datetime

output = {
    'ccos_version':      'bridge_v1',
    'evaluated_at':      datetime.utcnow().isoformat() + 'Z',
    'format_id':         rubric.format_id,
    'platform':          PLATFORM,
    'intent_signal':     INTENT_SIGNAL,
    'caption_copy':      CAPTION_COPY,
    'ccs':               raw_ccs,
    'threshold_label':   threshold_label,
    'threshold_action':  threshold_action,
    'consensus_warning': consensus['consensus_warning'],
    'consensus_detail':  consensus,
    'persona_results':   [
        {k: v for k, v in r.items() if not k.startswith('_')}
        for r in persona_results
    ]
}

fname = f'ccos_result_{datetime.utcnow().strftime("%Y%m%d_%H%M%S")}.json'
with open(fname, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Results saved to: {fname}')
print(json.dumps(output, indent=2)[:800] + '\n...')

---
## What this notebook proves + what comes next

**What v2 demonstrates (in addition to v1):**
- The rubric registry now contains three production-ready rubrics grounded in Phase 3 Deep Research
- YouTube Long-Form rubric: five dimensions with MBD z-score benchmarks, three-surface alignment, brand safety pre-gate
- YouTube Shorts rubric: five dimensions with VVSA benchmarks, completion rate formula, loop engineering, compliance pre-gate
- The `pre_scoring_gate` field on `FormatRubric` implements the binary compliance check before dimension scoring
- `route_to_rubric()` and `check_pre_scoring_gate()` are the two entry points for any new format evaluation

**Known limitations of this Bridge evaluation:**
- Personas are constructed from brief context — not grounded in client first-party data
- Benchmark calibration requires the Alpha session (20+ real rejected assets)
- Video evaluation uses still frames — the full video pipeline (FFmpeg keyframe extraction at 0s, 3s, 10s) is a Phase 2 build

**Phase 3 research in progress:**
- Meta (Facebook + Instagram) — next platform
- LinkedIn
- TikTok (reference)
- Each platform's rubric slots into `RUBRIC_REGISTRY` when research is complete

**To add a new format rubric (e.g. Meta Feed):**
```python
META_FEED_RUBRIC = FormatRubric(
    format_id   = 'meta_feed',
    format_name = 'Meta Feed (Facebook + Instagram)',
    modality    = 'vision',
    pre_scoring_gate = None,  # or define a gate if research warrants one
    platform_context = '...',
    dimensions  = [...]
)
RUBRIC_REGISTRY['meta_feed'] = META_FEED_RUBRIC
```
No other code changes required.
